## Phân tích chi tiết Observation Vector của Robot H1 Unitree trong Isaac Sim

## 1. Base Linear Velocity (3 chiều)

**Nguồn**: `base_lin_vel` - Vận tốc tuyến tính của base robot trong body frame

**Kích thước**: 3 (x, y, z)

**Mô tả**:
- Đo vận tốc di chuyển của thân robot theo 3 trục
- Được biểu diễn trong hệ tọa độ body frame (local frame của robot)
- **Noise**: Uniform noise [-0.1, 0.1] được thêm vào để tăng tính robust

**Ý nghĩa**:
- Giúp policy biết robot đang di chuyển nhanh/chậm như thế nào
- Quan trọng cho việc điều khiển vận tốc và ổn định

**Cấu hình trong h1_env.yaml**:
```yaml
base_lin_vel:
  func: omni.isaac.lab.envs.mdp.observations:base_lin_vel
  noise:
    func: omni.isaac.lab.utils.noise.noise_model:uniform_noise
    operation: add
    n_min: -0.1
    n_max: 0.1
```

## 2. Base Angular Velocity (3 chiều)

**Nguồn**: `base_ang_vel` - Vận tốc góc của base robot trong body frame

**Kích thước**: 3 (roll, pitch, yaw rates)

**Mô tả**:
- Đo tốc độ quay của thân robot quanh 3 trục
- Được biểu diễn trong body frame
- **Noise**: Uniform noise [-0.2, 0.2] (lớn hơn linear velocity vì góc nhạy cảm hơn)

**Ý nghĩa**:
- Giúp policy duy trì cân bằng và ổn định
- Phát hiện khi robot bị nghiêng hoặc quay không mong muốn
- Quan trọng cho việc điều khiển hướng di chuyển

**Cấu hình trong h1_env.yaml**:
```yaml
base_ang_vel:
  func: omni.isaac.lab.envs.mdp.observations:base_ang_vel
  noise:
    func: omni.isaac.lab.utils.noise.noise_model:uniform_noise
    operation: add
    n_min: -0.2
    n_max: 0.2
```

## 3. Projected Gravity (3 chiều)

**Nguồn**: `projected_gravity` - Vector trọng lực được chiếu vào body frame

**Kích thước**: 3 (gx, gy, gz trong body frame)

**Mô tả**:
- Vector trọng lực (0, 0, -9.81) trong world frame được chiếu vào body frame của robot
- Cho biết hướng "xuống" tương đối với robot
- **Noise**: Uniform noise [-0.05, 0.05]

**Ý nghĩa**:
- **Cực kỳ quan trọng** cho việc duy trì cân bằng
- Giúp policy biết robot đang nghiêng về phía nào
- Thay thế cho việc sử dụng trực tiếp orientation (quaternion)
- Ví dụ: Nếu robot đứng thẳng → projected_gravity ≈ (0, 0, -9.81)
- Nếu robot nghiêng về trước → projected_gravity có thành phần x âm

**Cấu hình trong h1_env.yaml**:
```yaml
projected_gravity:
  func: omni.isaac.lab.envs.mdp.observations:projected_gravity
  noise:
    func: omni.isaac.lab.utils.noise.noise_model:uniform_noise
    operation: add
    n_min: -0.05
    n_max: 0.05
```

## 4. Velocity Commands (3 chiều)

**Nguồn**: `velocity_commands` - Lệnh vận tốc mục tiêu từ command generator

**Kích thước**: 3 (vx_cmd, vy_cmd, vyaw_cmd)

**Mô tả**:
- Vận tốc mục tiêu mà robot cần đạt được
- Được sinh ra từ `UniformVelocityCommand` với các range:
  - `lin_vel_x`: [0.0, 1.0] m/s (chỉ đi tiến)
  - `lin_vel_y`: [0.0, 0.0] m/s (không đi ngang)
  - `ang_vel_z`: [-1.0, 1.0] rad/s (quay trái/phải)
- **Không có noise** (vì đây là command, không phải measurement)
- Resampling mỗi 10 giây

**Ý nghĩa**:
- Cho policy biết mục tiêu cần đạt được
- Policy sẽ học cách điều khiển robot để tracking các command này
- 2% môi trường là standing (vx=vy=vyaw=0)

**Cấu hình trong h1_env.yaml**:
```yaml
velocity_commands:
  func: omni.isaac.lab.envs.mdp.observations:generated_commands
  params:
    command_name: base_velocity
  noise: null

commands:
  base_velocity:
    class_type: omni.isaac.lab.envs.mdp.commands.velocity_command:UniformVelocityCommand
    resampling_time_range: [10.0, 10.0]
    ranges:
      lin_vel_x: [0.0, 1.0]
      lin_vel_y: [0.0, 0.0]
      ang_vel_z: [-1.0, 1.0]
    rel_standing_envs: 0.02
```

## 5. Joint Positions (19 chiều)

**Nguồn**: `joint_pos` - Vị trí góc của các khớp (relative to default position)

**Kích thước**: 19 joints

**Danh sách các khớp của H1**:
- **Chân (10 joints)**:
  - Mỗi chân: hip_yaw, hip_roll, hip_pitch, knee, ankle (5 joints × 2 = 10)
- **Thân (1 joint)**:
  - torso (khớp thắt lưng)
- **Tay (8 joints)**:
  - Mỗi tay: shoulder_pitch, shoulder_roll, shoulder_yaw, elbow (4 joints × 2 = 8)

**Mô tả**:
- Sử dụng `joint_pos_rel` - vị trí tương đối so với default position
- **Noise**: Uniform noise [-0.01, 0.01] rad
- Default positions được định nghĩa trong init_state

**Ý nghĩa**:
- Cho policy biết tư thế hiện tại của robot
- Quan trọng cho việc tạo gait pattern (chu kỳ bước đi)
- Giúp tránh va chạm giữa các phần của robot

**Cấu hình trong h1_env.yaml**:
```yaml
joint_pos:
  func: omni.isaac.lab.envs.mdp.observations:joint_pos_rel
  noise:
    func: omni.isaac.lab.utils.noise.noise_model:uniform_noise
    operation: add
    n_min: -0.01
    n_max: 0.01

# Default joint positions
init_state:
  joint_pos:
    .*_hip_yaw: 0.0
    .*_hip_roll: 0.0
    .*_hip_pitch: -0.28
    .*_knee: 0.79
    .*_ankle: -0.52
    torso: 0.0
    .*_shoulder_pitch: 0.28
    .*_shoulder_roll: 0.0
    .*_shoulder_yaw: 0.0
    .*_elbow: 0.52
```

## 6. Joint Velocities (19 chiều)

**Nguồn**: `joint_vel` - Vận tốc góc của các khớp

**Kích thước**: 19 joints (tương ứng với joint positions)

**Mô tả**:
- Sử dụng `joint_vel_rel` - vận tốc tương đối
- **Noise**: Uniform noise [-1.5, 1.5] rad/s (noise lớn vì velocity khó đo chính xác)

**Ý nghĩa**:
- Cho policy biết các khớp đang chuyển động nhanh/chậm như thế nào
- Quan trọng cho việc điều khiển mượt mà và tránh rung động
- Giúp policy dự đoán trạng thái tiếp theo

**Cấu hình trong h1_env.yaml**:
```yaml
joint_vel:
  func: omni.isaac.lab.envs.mdp.observations:joint_vel_rel
  noise:
    func: omni.isaac.lab.utils.noise.noise_model:uniform_noise
    operation: add
    n_min: -1.5
    n_max: 1.5
```

---

## 7. Last Actions (19 chiều)

**Nguồn**: `actions` - Action từ timestep trước đó

**Kích thước**: 19 (tương ứng với số lượng joints được điều khiển)

**Mô tả**:
- Lưu lại action mà policy đã output ở bước trước
- **Không có noise** (vì đây là action đã thực hiện, không phải measurement)
- Action type: `JointPositionAction` với scale=0.5

**Ý nghĩa**:
- Giúp policy có "memory" về hành động vừa thực hiện
- Quan trọng cho việc tạo chuyển động mượt mà (smooth transitions)
- Tránh thay đổi action đột ngột giữa các timesteps
- Giúp policy học được temporal patterns

**Cấu hình trong h1_env.yaml**:
```yaml
actions:
  func: omni.isaac.lab.envs.mdp.observations:last_action
  noise: null

# Action configuration
actions:
  joint_pos:
    class_type: omni.isaac.lab.envs.mdp.actions.joint_actions:JointPositionAction
    asset_name: robot
    joint_names: [".*"]
    scale: 0.5
    offset: 0.0
    use_default_offset: true
```

---

## Tổng kết Observation Vector

### Kích thước tổng cộng: **69 chiều**

| Thành phần | Kích thước | Noise | Mục đích chính |
|------------|-----------|-------|----------------|
| Base Linear Velocity | 3 | ±0.1 | Tracking vận tốc di chuyển |
| Base Angular Velocity | 3 | ±0.2 | Duy trì cân bằng và hướng |
| Projected Gravity | 3 | ±0.05 | Phát hiện độ nghiêng |
| Velocity Commands | 3 | None | Mục tiêu cần đạt |
| Joint Positions | 19 | ±0.01 | Tư thế hiện tại |
| Joint Velocities | 19 | ±1.5 | Động lực học khớp |
| Last Actions | 19 | None | Memory và smooth control |
| **TỔNG** | **69** | | |

### Đặc điểm quan trọng:

1. **Concatenation**: Tất cả được nối thành 1 vector duy nhất (concatenate_terms: true)
2. **Noise Injection**: Enable corruption để tăng robustness khi deploy lên robot thật
3. **Không có Height Scanner**: Môi trường này chỉ dùng flat terrain, không cần scan địa hình
4. **Proprioceptive Only**: Chỉ dùng thông tin nội tại (không có camera, lidar)

### So sánh với các task khác:

**H1 Locomotion (file này)**:
- 69 chiều
- Focus: Đi bộ trên mặt phẳng
- Không có camera, không có object

**H1-2 Pick-Place Tasks** (trong unitree_sim_isaaclab):
- Observation phức tạp hơn nhiều
- Bao gồm: robot_joint_state (26×3=78 chiều) + inspire_state (12 chiều) + camera_image
- Focus: Manipulation với tay và gripper

In [ ]:
# Detailed breakdown of observation vector indices
import pandas as pd

# Create detailed observation breakdown
obs_breakdown = []
start_idx = 0

components_detail = [
    ("Base Linear Velocity", ["vx", "vy", "vz"], 3, "±0.1"),
    ("Base Angular Velocity", ["wx", "wy", "wz"], 3, "±0.2"),
    ("Projected Gravity", ["gx", "gy", "gz"], 3, "±0.05"),
    ("Velocity Commands", ["vx_cmd", "vy_cmd", "vyaw_cmd"], 3, "None"),
    ("Joint Positions", ["19 joints"], 19, "±0.01"),
    ("Joint Velocities", ["19 joints"], 19, "±1.5"),
    ("Last Actions", ["19 actions"], 19, "None"),
]

for comp_name, sub_names, dim, noise in components_detail:
    end_idx = start_idx + dim
    obs_breakdown.append({
        "Component": comp_name,
        "Index Range": f"[{start_idx}:{end_idx}]",
        "Dimensions": dim,
        "Sub-components": ", ".join(sub_names),
        "Noise": noise
    })
    start_idx = end_idx

df = pd.DataFrame(obs_breakdown)
print("=" * 100)
print("H1 OBSERVATION VECTOR BREAKDOWN")
print("=" * 100)
print(df.to_string(index=False))
print("=" * 100)
print(f"Total dimensions: {start_idx}")
print("=" * 100)

---

## So sánh với H1-2 Manipulation Tasks

### H1 Locomotion (file này) vs H1-2 Pick-Place

| Aspect | H1 Locomotion | H1-2 Pick-Place (Manipulation) |
|--------|---------------|--------------------------------|
| **Observation Size** | 69 chiều | ~90+ chiều (không concatenate) |
| **Robot Joints** | 19 joints | 26 joints (body) + 12 joints (inspire gripper) |
| **Observation Components** | - base_lin_vel (3)<br>- base_ang_vel (3)<br>- projected_gravity (3)<br>- velocity_commands (3)<br>- joint_pos (19)<br>- joint_vel (19)<br>- last_actions (19) | - robot_joint_state (26×3=78)<br>- robot_inspire_state (12)<br>- camera_image (RGB) |
| **Sensors** | Proprioceptive only | Proprioceptive + Vision |
| **Camera** | None | 3 cameras (front, left wrist, right wrist) |
| **Task Focus** | Walking/Locomotion | Manipulation (pick & place) |
| **Action Space** | 19 (joint positions) | 26 (body + arms) |
| **Concatenation** | True (single vector) | False (dict of observations) |
| **Environment** | Flat terrain | Table with objects |

### Key Differences:

1. **Complexity**: Manipulation tasks cần nhiều thông tin hơn (vision, gripper state)
2. **Joint Count**: H1-2 có thêm 7 joints cho gripper (26 body + 12 inspire = 38 total)
3. **Observation Structure**: 
   - Locomotion: Concatenated vector → dễ dùng với MLP
   - Manipulation: Dictionary → cần multimodal network (CNN + MLP)
4. **Control Frequency**: Cả hai đều 50 Hz (decimation=4, dt=0.005s)

In [ ]:
# Summary visualization: Compare H1 Locomotion vs H1-2 Manipulation
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Observation dimensions comparison
tasks = ['H1\nLocomotion', 'H1-2\nManipulation']
obs_dims = [69, 90]
colors_task = ['#3498db', '#e74c3c']

axes[0].bar(tasks, obs_dims, color=colors_task, alpha=0.7, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Observation Dimensions', fontsize=12, fontweight='bold')
axes[0].set_title('Observation Size Comparison', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(obs_dims):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold', fontsize=11)

# 2. Joint count comparison
joint_categories = ['Body\nJoints', 'Gripper\nJoints', 'Total\nJoints']
h1_joints = [19, 0, 19]
h12_joints = [26, 12, 38]

x = np.arange(len(joint_categories))
width = 0.35

bars1 = axes[1].bar(x - width/2, h1_joints, width, label='H1 Locomotion', color='#3498db', alpha=0.7, edgecolor='black')
bars2 = axes[1].bar(x + width/2, h12_joints, width, label='H1-2 Manipulation', color='#e74c3c', alpha=0.7, edgecolor='black')

axes[1].set_ylabel('Number of Joints', fontsize=12, fontweight='bold')
axes[1].set_title('Joint Count Comparison', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(joint_categories)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            axes[1].text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# 3. Sensor modalities
modalities = ['Proprioception', 'Vision', 'Force']
h1_sensors = [1, 0, 0]  # Only proprioception
h12_sensors = [1, 1, 0]  # Proprioception + Vision

x = np.arange(len(modalities))
bars1 = axes[2].bar(x - width/2, h1_sensors, width, label='H1 Locomotion', color='#3498db', alpha=0.7, edgecolor='black')
bars2 = axes[2].bar(x + width/2, h12_sensors, width, label='H1-2 Manipulation', color='#e74c3c', alpha=0.7, edgecolor='black')

axes[2].set_ylabel('Used (1) / Not Used (0)', fontsize=12, fontweight='bold')
axes[2].set_title('Sensor Modalities', fontsize=13, fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels(modalities)
axes[2].set_ylim([0, 1.2])
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("SUMMARY: H1 Observation Vector Analysis")
print("="*80)
print(f"✓ Total observation dimensions: 69")
print(f"✓ Components: 7 (base velocities, gravity, commands, joints, actions)")
print(f"✓ Noise injection: Enabled for sim-to-real transfer")
print(f"✓ Network: Actor-Critic with 3×128 hidden layers")
print(f"✓ Training: PPO with 4096 parallel environments")
print(f"✓ Control frequency: 50 Hz (decimation=4)")
print("="*80)

---

## Policy Network Architecture

Dựa trên file `agent.yaml`, policy network có cấu trúc:

```yaml
policy:
  class_name: ActorCritic
  init_noise_std: 1.0
  actor_hidden_dims: [128, 128, 128]
  critic_hidden_dims: [128, 128, 128]
  activation: elu
```

**Actor Network**:
- Input: 69 chiều (observation vector)
- Hidden layers: 3 layers × 128 neurons
- Activation: ELU
- Output: 19 chiều (joint position actions)

**Critic Network**:
- Input: 69 chiều (observation vector)
- Hidden layers: 3 layers × 128 neurons
- Activation: ELU
- Output: 1 chiều (value estimate)

**Training Algorithm**: PPO (Proximal Policy Optimization)
- Learning rate: 0.001
- Clip param: 0.2
- Entropy coef: 0.01
- Gamma: 0.99
- Lambda (GAE): 0.95

---

## Reward Function

Reward được thiết kế để khuyến khích robot đi bộ mượt mà và ổn định:

### Positive Rewards:
1. **track_lin_vel_xy_exp** (weight: 1.0): Tracking vận tốc tuyến tính xy
2. **track_ang_vel_z_exp** (weight: 1.0): Tracking vận tốc góc z
3. **feet_air_time** (weight: 1.0): Khuyến khích thời gian chân không chạm đất (gait)

### Negative Penalties:
1. **ang_vel_xy_l2** (weight: -0.05): Phạt quay quanh x, y (giữ thẳng)
2. **dof_acc_l2** (weight: -1.25e-07): Phạt gia tốc khớp cao
3. **action_rate_l2** (weight: -0.005): Phạt thay đổi action nhanh
4. **flat_orientation_l2** (weight: -1.0): Phạt nghiêng thân
5. **dof_pos_limits** (weight: -1.0): Phạt vượt giới hạn khớp ankle
6. **feet_slide** (weight: -0.25): Phạt trượt chân
7. **joint_deviation_hip** (weight: -0.2): Phạt hip lệch khỏi default
8. **joint_deviation_arms** (weight: -0.2): Phạt tay lệch khỏi default
9. **joint_deviation_torso** (weight: -0.1): Phạt torso lệch khỏi default
10. **termination_penalty** (weight: -200.0): Phạt nặng khi ngã

---

## Simulation Settings

- **Physics timestep**: 0.005s (200 Hz)
- **Decimation**: 4 → Control frequency: 50 Hz
- **Episode length**: 20 seconds
- **Number of environments**: 4096 (parallel training)
- **Device**: CUDA (GPU acceleration)